# SymPy — symbolic mathematics in Python

**What this library does in one sentence:** SymPy treats math like algebra class — variables stay as symbols (`x`, `θ`), so you can differentiate, integrate, simplify, and rearrange expressions *exactly*, then export the result as fast NumPy code.

For the cart-pole project, SymPy is how we derive the equations of motion symbolically — once, verifiably — and then hand them to the numerical simulator.

In [ ]:
import sympy as sp
import numpy as np

## 1. Symbolic vs numerical — the `sqrt(2)` example

Numerically, `np.sqrt(2)` gives `1.4142135623730951` — a `float64` approximation.

Symbolically, `sp.sqrt(2)` gives `sqrt(2)` — the *exact* object. If you later compute `sp.sqrt(2)**2`, you get back the integer `2`, not `2.0000000000000004`.

In [ ]:
print('numerical:', np.sqrt(2)**2)     # 2.0000000000000004 — float error
print('symbolic: ', sp.sqrt(2)**2)      # 2 — exact

**The point:** when you're *deriving* something (taking derivatives, solving equations, simplifying), you want symbolic. When you're *simulating* something fast (10,000 timesteps), you want numerical. SymPy lets you do both: derive symbolically, export numerically.

## 2. Core symbol types

### `sp.Symbol('x')` vs `sp.symbols('x y z')`

| Function | What it returns |
|----------|-----------------|
| `sp.Symbol('x')`            | One symbol. |
| `sp.symbols('x y z')`       | A tuple of symbols (space- or comma-separated names). |

In [ ]:
x = sp.Symbol('x')
y, z = sp.symbols('y z')        # tuple unpacking
print(x + y + z)

# pendulum — declare the parameters
M, m, L, g, t = sp.symbols('M m L g t')
print(M, m, L, g, t)

### Assumptions: `positive=True`, `real=True`

By default, SymPy doesn't assume anything about a symbol — it could be complex, negative, zero. That makes simplification cautious. Telling SymPy a symbol is positive or real unlocks better simplification.

In [ ]:
a = sp.Symbol('a')
print(sp.sqrt(a**2))           # |a| — because a could be negative

a = sp.Symbol('a', positive=True)
print(sp.sqrt(a**2))           # a — clean

# pendulum — masses and lengths are always positive
M, m, L, g = sp.symbols('M m L g', positive=True)

**Why it matters here:** denominators in the cart-pole EOM include `M + m*sin²(θ)`. SymPy can simplify these only if it knows `M, m` are positive.

### `sp.Function('f')(t)` — functions of time

Most EOM derivations need *functions* (not just variables) so that you can differentiate them. `θ(t)` is a function — its derivative is `θ̇(t)`.

In [ ]:
t = sp.symbols('t')
x_t  = sp.Function('x')(t)        # the cart position as a function of time
th_t = sp.Function('theta')(t)    # the pole angle as a function of time

print(x_t)
print(sp.diff(x_t, t))            # Derivative(x(t), t)  — exactly what we want

**Pendulum use:** the Lagrangian needs `θ(t)`. After taking derivatives we'll substitute `θ(t) → θ`, `θ̇(t) → θ̇`, etc., to get a clean algebraic equation.

### `sp.Rational(1, 2)` vs `0.5`

`0.5` is a Python `float` — SymPy will silently convert it. Once a float is in your expression, exact simplification stops working.

In [ ]:
x = sp.Symbol('x')
print(sp.Rational(1, 2) * x + sp.Rational(1, 2) * x)   # x  — clean
print(0.5 * x + 0.5 * x)                               # 1.0*x  — already polluted with floats

**Rule:** in *symbolic* code, use `sp.Rational(1, 2)`, `sp.Rational(3, 4)`, never `0.5` or `0.75`. The kinetic energy `½ m v²` becomes `sp.Rational(1, 2) * m * v**2`.

## 3. Basic operations

### `sp.diff(expr, var)` — differentiation

| Parameter | Meaning |
|-----------|---------|
| `expr`    | The expression to differentiate. |
| `var`     | The variable to differentiate with respect to. |
| (extra)   | Pass an integer for higher derivatives: `sp.diff(expr, var, 2)` = second derivative. |

In [ ]:
x = sp.Symbol('x')
f = x**3 + sp.sin(x)
print(sp.diff(f, x))           # 3*x**2 + cos(x)
print(sp.diff(f, x, 2))        # 6*x - sin(x)  — second derivative

# pendulum — kinetic energy of the bob, differentiated by time
t    = sp.symbols('t')
m    = sp.symbols('m', positive=True)
x_t  = sp.Function('x')(t)
T    = sp.Rational(1, 2) * m * sp.diff(x_t, t)**2
print(T)
print(sp.diff(T, t))           # rate of change of KE

### `sp.integrate(expr, var)`

Indefinite integral (no constant) or definite integral with a tuple `(var, lo, hi)`.

In [ ]:
x = sp.Symbol('x')
print(sp.integrate(sp.cos(x), x))                # sin(x)
print(sp.integrate(x**2, (x, 0, 1)))             # 1/3 — definite, from 0 to 1

### `sp.solve` — solving equations

Solves `expr = 0` (or a list of equations) for the given variables.

| Parameter | Meaning |
|-----------|---------|
| `eq`      | Expression set equal to zero (or `sp.Eq(lhs, rhs)`). |
| `var(s)`  | Single variable or list of variables to solve for. |

In [ ]:
x, y = sp.symbols('x y')
print(sp.solve(x**2 - 4, x))                              # [-2, 2]
print(sp.solve([x + y - 3, x - y - 1], [x, y]))           # {x: 2, y: 1}

# pendulum — given a 2x2 mass-matrix equation M*qdd = b, solve for qdd
M_mat = sp.Matrix([[2, 1], [1, 3]])
b_vec = sp.Matrix([5, 11])
qdd1, qdd2 = sp.symbols('qdd1 qdd2')
sol = sp.solve(M_mat * sp.Matrix([qdd1, qdd2]) - b_vec, [qdd1, qdd2])
print(sol)

### `sp.simplify`, `sp.expand`, `sp.factor`, `sp.trigsimp`, `sp.collect`

All take an expression and try to massage it. They don't change what the expression *means*, only how it *looks*.

| Function | What it does |
|----------|--------------|
| `simplify(e)`   | Try a bunch of heuristics — "make it look smaller." Slow. |
| `expand(e)`     | Multiply everything out. `(x+1)**2` → `x**2 + 2*x + 1`. |
| `factor(e)`     | Factor a polynomial. `x**2 + 2*x + 1` → `(x+1)**2`. |
| `trigsimp(e)`   | Apply trig identities. `sin² + cos²` → `1`. |
| `collect(e, x)` | Group terms by powers of `x`. |

In [ ]:
x, a, b = sp.symbols('x a b')
th = sp.Symbol('theta')

print(sp.expand((x + 1)**3))
print(sp.factor(x**2 - 1))
print(sp.trigsimp(sp.sin(th)**2 + sp.cos(th)**2))    # 1
print(sp.collect(a*x + b*x + a, x))                  # x*(a + b) + a
print(sp.simplify((x**2 - 1)/(x + 1)))               # x - 1

**For the EOM:** after taking the Lagrangian derivatives you'll get a ugly mess of `sin`s and `cos`s. `trigsimp` will collapse identities, then `collect(eq, qdd)` groups by the second derivatives, ready to solve.

## 4. Substitution: `.subs({old: new})`

Replaces symbols (or sub-expressions) in an expression. Used **constantly** — to plug in parameter values, to swap `θ(t)` for `θ`, to test specific cases.

In [ ]:
x, a = sp.symbols('x a')
expr = a * x**2

print(expr.subs(x, 3))                       # 9*a
print(expr.subs({x: 3, a: 2}))               # 18

# pendulum — collapse  theta(t) → theta,  Derivative(theta(t), t) → thd
t    = sp.symbols('t')
th_t = sp.Function('theta')(t)
th, thd = sp.symbols('theta thd')

expr  = sp.diff(th_t, t)**2 + sp.sin(th_t)
clean = expr.subs({sp.diff(th_t, t): thd, th_t: th})
print(clean)                                  # thd**2 + sin(theta)

**Critical for lambdify:** before exporting an expression to a NumPy function, you *must* `subs` all the `Function`/`Derivative` objects to plain symbols. `lambdify` cannot turn `Derivative(theta(t), t)` into a NumPy expression.

## 5. Printing

### `sp.pprint(expr)` — pretty print

Renders the expression in ASCII (or unicode) math layout. Much easier to read than the default `str(...)` form.

In [ ]:
x = sp.Symbol('x')
expr = (x + 1)**2 / (x - 1)
sp.pprint(expr)

### `sp.latex(expr)` — LaTeX source

Useful when you want to paste the derived equation into a notebook, a report, or a Manim animation.

In [ ]:
print(sp.latex(sp.Rational(1, 2) * sp.Symbol('m') * sp.Symbol('v')**2))
# → \frac{m v^{2}}{2}

## 6. `sp.lambdify` — the bridge to NumPy

This is **the** function that lets you derive symbolically and then simulate numerically. It compiles a SymPy expression into a fast Python function that takes NumPy arrays.

```python
f = sp.lambdify(args, expr, 'numpy')
```

| Parameter | Meaning |
|-----------|---------|
| `args`    | Tuple/list of symbols that will become the function's parameters. |
| `expr`    | The SymPy expression (or list of expressions) to compile. |
| `'numpy'` | Backend — tells `lambdify` to use NumPy functions (so arrays work). |

In [ ]:
x, y = sp.symbols('x y')
expr = x**2 + sp.sin(y)

f = sp.lambdify((x, y), expr, 'numpy')

print(f(3, 0))                              # 9.0
print(f(np.array([1, 2, 3]), 0))            # [1. 4. 9.] — broadcasts over arrays

### Common mistake — forgetting to `subs` derivatives first

If you `lambdify` an expression that still contains `Derivative(theta(t), t)`, you'll get a function that takes `Derivative` *objects* — totally useless numerically.

Always `subs` derivatives and time-functions to plain symbols before lambdifying.

In [ ]:
t    = sp.symbols('t')
th_t = sp.Function('theta')(t)
th, thd = sp.symbols('theta thd')

# imagine this came out of a Lagrangian derivation
raw = sp.diff(th_t, t)**2 + sp.cos(th_t)

# WRONG — this won't work numerically
# bad = sp.lambdify((th_t, sp.diff(th_t, t)), raw, 'numpy')

# RIGHT — substitute first, then lambdify the clean expression
clean = raw.subs({sp.diff(th_t, t): thd, th_t: th})
good  = sp.lambdify((th, thd), clean, 'numpy')
print(good(0.5, 1.2))

## 7. Applied: deriving a simple pendulum EOM via Lagrangian

We'll derive $\ddot{\theta} = -\frac{g}{L}\sin\theta$ symbolically. This is the warm-up for the full cart-pole derivation.

In [ ]:
# 1. declare symbols and the angle-as-a-function-of-time
t          = sp.symbols('t')
m, L, g    = sp.symbols('m L g', positive=True)
theta      = sp.Function('theta')(t)

# 2. position of the bob (origin at the pivot, x right, y up)
x_bob =  L * sp.sin(theta)
y_bob = -L * sp.cos(theta)

# 3. velocity of the bob
xdot = sp.diff(x_bob, t)
ydot = sp.diff(y_bob, t)

# 4. kinetic and potential energy
T = sp.Rational(1, 2) * m * (xdot**2 + ydot**2)
T = sp.simplify(T)
V = m * g * y_bob

# Lagrangian L = T - V
Lag = T - V
sp.pprint(Lag)

In [ ]:
# 5. Euler-Lagrange:  d/dt (dL/d(theta_dot))  -  dL/d(theta)  =  0
theta_dot = sp.diff(theta, t)

dL_dthd     = sp.diff(Lag, theta_dot)
ddt_dL_dthd = sp.diff(dL_dthd, t)
dL_dth      = sp.diff(Lag, theta)

EL = sp.simplify(ddt_dL_dthd - dL_dth)
sp.pprint(EL)        # should be:  L*m*(L*theta_dd + g*sin(theta))

In [ ]:
# 6. solve for theta_ddot
theta_ddot = sp.diff(theta, t, 2)
sol = sp.solve(EL, theta_ddot)
print('theta_ddot =', sol[0])     # -g*sin(theta)/L

In [ ]:
# 7. substitute time-functions to plain symbols, then lambdify
th_sym, thd_sym = sp.symbols('th thd')
rhs_clean = sol[0].subs({theta: th_sym, theta_dot: thd_sym})

f = sp.lambdify((th_sym, thd_sym, L, g), rhs_clean, 'numpy')

# now it's a plain numerical function — use it with solve_ivp or RK4
print('theta_ddot at theta=0.1, g=9.81, L=1:', f(0.1, 0.0, 1.0, 9.81))

**For the cart-pole:** repeat this same recipe with two generalised coordinates `x(t)` and `θ(t)`, derive `T` and `V` for both cart and bob, take the Euler-Lagrange equations for each coordinate, solve for `[ẍ, θ̈]`, and lambdify. That's exactly what `Day_1/pendulum1.ipynb` does.

## Functions used in this project — quick reference

| Function | Signature | What it does |
|----------|-----------|--------------|
| `sp.Symbol`     | `sp.Symbol('x', positive=True)` | One symbol with optional assumptions. |
| `sp.symbols`    | `sp.symbols('a b c')`           | Multiple symbols at once. |
| `sp.Function`   | `sp.Function('f')(t)`            | A function of `t` — needed for time-derivatives. |
| `sp.Rational`   | `sp.Rational(1, 2)`              | Exact fraction — use instead of `0.5`. |
| `sp.diff`       | `sp.diff(expr, var[, n])`        | Differentiate (optionally `n` times). |
| `sp.integrate`  | `sp.integrate(expr, var)` or `(var, lo, hi)` | Indefinite or definite integral. |
| `sp.solve`      | `sp.solve(eq, var)` or `solve([eqs], [vars])` | Solve `eq = 0`. |
| `sp.simplify`   | `sp.simplify(expr)`              | Generic simplification. |
| `sp.expand`     | `sp.expand(expr)`                | Multiply everything out. |
| `sp.factor`     | `sp.factor(expr)`                | Factor polynomials. |
| `sp.trigsimp`   | `sp.trigsimp(expr)`              | Apply trig identities. |
| `sp.collect`    | `sp.collect(expr, var)`          | Group terms by `var`. |
| `.subs`         | `expr.subs({old: new, ...})`     | Replace symbols / sub-expressions. |
| `sp.pprint`     | `sp.pprint(expr)`                | Pretty-print to terminal. |
| `sp.latex`      | `sp.latex(expr)`                 | Return LaTeX source. |
| `sp.lambdify`   | `sp.lambdify(args, expr, 'numpy')` | Compile to a NumPy-compatible function. |